# ViPIBench Internal Experiment Workflow

## Purpose

This notebook documents the lower-level engineering workflow for dataset reconstruction, baseline evaluation, detector training, model selection, target-agent execution, and adaptive attack evaluation. It is retained as an internal, checkpoint-aware workflow and is not the operator-facing Colab launch entry point.

## Execution boundary

The default `smoke` mode validates the local workflow contract without initiating confirmatory model execution. Confirmatory mode is fail-closed: it requires explicit approval through `VIPIBENCH_CONFIRMATORY_RUN_APPROVED=YES` and an observed runtime that satisfies the declared accelerator profile. The notebook does not mount external storage, upload artifacts, publish results, or authorize public release.

Outputs from this workflow constitute engineering or experimental evidence only within their recorded configuration, hashes, and split definitions. Statistical conclusions and thesis claims require the separate frozen analysis and final claim-audit stages.

In [ ]:
import json
import os
from pathlib import Path

MODE = os.environ.get("VIPIBENCH_MODE", "smoke")
PROJECT_ROOT = Path(os.environ.get("VIPIBENCH_PROJECT_ROOT", ".")).resolve()
dataset_default = PROJECT_ROOT / "data" / "processed" / "vipibench_exec.jsonl"
split_default = PROJECT_ROOT / "data" / "splits" / "frozen"
output_default = PROJECT_ROOT / "outputs" / "latest"
DATASET_PATH = Path(os.environ.get("VIPIBENCH_DATASET_PATH", dataset_default)).resolve()
SPLIT_DIR = Path(os.environ.get("VIPIBENCH_SPLIT_DIR", split_default)).resolve()
OUTPUT_ROOT = Path(os.environ.get("VIPIBENCH_OUTPUT_ROOT", output_default)).resolve()
if MODE not in {"smoke", "confirmatory"}:
    raise ValueError("VIPIBENCH_MODE must be smoke or confirmatory")
print({"mode": MODE, "project_root": str(PROJECT_ROOT), "output_root": str(OUTPUT_ROOT)})

In [ ]:
import subprocess

INSTALL_COMMAND = ["python", "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[experiment]"]
subprocess.check_call(INSTALL_COMMAND)

In [ ]:
if MODE == "smoke":
    smoke_contract = [
        PROJECT_ROOT / "pyproject.toml",
        PROJECT_ROOT / "requirements-experiment.lock",
        DATASET_PATH,
        SPLIT_DIR / "split_manifest.json",
    ]
    missing = [str(path) for path in smoke_contract if not path.is_file()]
    if missing:
        raise FileNotFoundError({"missing_smoke_contract_paths": missing})
    print({"status": "PASS", "mode": "smoke", "scope": "structural_only"})
else:
    from vipibench.readiness import evaluate_launch_readiness

    readiness = evaluate_launch_readiness(PROJECT_ROOT, require_clean_environment=False)
    if readiness["status"] != "PASS":
        raise RuntimeError(readiness["failed_checks"])
    print(readiness["milestone"])

In [ ]:
from vipibench.dataio import write_json
from vipibench.runtime_capacity import check_runtime_profile_path

RUNTIME_PROFILE = PROJECT_ROOT / "configs" / "profiles" / "accelerator_80gb.yaml"
RUNTIME_COMMAND = ["vipibench", "check-runtime", "--profile", str(RUNTIME_PROFILE)]
if MODE == "confirmatory":
    if os.environ.get("VIPIBENCH_CONFIRMATORY_RUN_APPROVED") != "YES":
        raise PermissionError("Explicit authorization is required before confirmatory execution")
    runtime = check_runtime_profile_path(RUNTIME_PROFILE, PROJECT_ROOT)
    write_json(PROJECT_ROOT / "outputs" / "resource_measurement.json", runtime)
    if runtime["status"] != "PASS":
        raise RuntimeError(runtime["errors"])
    print("Observed runtime gate PASS", runtime["probe"])

In [ ]:
if MODE == "confirmatory":
    subprocess.check_call(["vipibench", "compile-provenance-contrast"])
    subprocess.check_call(["vipibench", "audit-provenance-contrast"])

In [ ]:
if MODE == "confirmatory":
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([
        "vipibench", "run-tfidf-baseline",
        "--project-root", str(PROJECT_ROOT),
        "--config", str(PROJECT_ROOT / "configs/models/tfidf_core.yaml"),
        "--output", str(OUTPUT_ROOT / "tfidf_baseline_manifest.json"),
    ])
    subprocess.check_call([
        "vipibench", "run-public-detector-benchmark",
        "--config", str(PROJECT_ROOT / "configs/models/public_detector.yaml"),
        "--split-dir", str(SPLIT_DIR),
        "--output-root", str(OUTPUT_ROOT / "public_detector"),
    ])
    subprocess.check_call([
        "vipibench", "run-encoder-matrix",
        "--config", str(PROJECT_ROOT / "configs/models/mdeberta_core.yaml"),
        "--split-dir", str(SPLIT_DIR),
        "--output-root", str(OUTPUT_ROOT / "mdeberta"),
    ])
    subprocess.check_call([
        "vipibench", "analyze-encoder-ablations",
        "--output-root", str(OUTPUT_ROOT / "mdeberta"),
        "--output", str(OUTPUT_ROOT / "encoder_ablation_analysis.json"),
    ])

In [ ]:
if MODE == "confirmatory":
    selection_path = OUTPUT_ROOT / "mdeberta" / "model_selection.json"
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    selected_run = selection["selected"]["run_id"]
    detector_model_version = selection["selected"]["model_artifact_version"]
    selected = OUTPUT_ROOT / "mdeberta" / selected_run
    print({"selected_run": selected_run, "selection_metric": selection["selection_metric"]})

In [ ]:
if MODE == "confirmatory":
    core_trajectories = OUTPUT_ROOT / "core_target_trajectories.jsonl"
    subprocess.check_call([
        "vipibench", "run-target-agent",
        "--dataset", str(SPLIT_DIR / "test.jsonl"),
        "--output", str(core_trajectories),
        "--checkpoint-dir", str(OUTPUT_ROOT / "core_target_checkpoints"),
    ])
    subprocess.check_call([
        "vipibench", "evaluate-four-arms",
        "--predictions", str(selected / "core_test_predictions.jsonl"),
        "--trajectories", str(core_trajectories),
        "--thresholds", str(selected / "thresholds.json"),
        "--detector-model-version", detector_model_version,
        "--test-dataset", str(SPLIT_DIR / "test.jsonl"),
        "--output", str(OUTPUT_ROOT / "static_four_arm_evaluation.json"),
    ])

In [ ]:
if MODE == "confirmatory":
    candidate_dataset = OUTPUT_ROOT / "attack_candidates.jsonl"
    candidate_scores = OUTPUT_ROOT / "attack_candidate_scores.jsonl"
    subprocess.check_call([
        "vipibench", "generate-attack-candidates",
        "--project-root", str(PROJECT_ROOT),
        "--detector-model-dir", str(selected / "model"),
        "--output-dataset", str(candidate_dataset),
        "--output-scores", str(candidate_scores),
        "--checkpoint-dir", str(OUTPUT_ROOT / "attack_search_checkpoints"),
    ])
    candidate_trajectories = OUTPUT_ROOT / "attack_target_trajectories.jsonl"
    subprocess.check_call([
        "vipibench", "run-target-agent",
        "--dataset", str(candidate_dataset),
        "--output", str(candidate_trajectories),
        "--checkpoint-dir", str(OUTPUT_ROOT / "attack_target_checkpoints"),
    ])
    subprocess.check_call([
        "vipibench", "evaluate-attack-search",
        "--candidate-dataset", str(candidate_dataset),
        "--candidate-scores", str(candidate_scores),
        "--target-trajectories", str(candidate_trajectories),
        "--thresholds", str(selected / "thresholds.json"),
        "--output", str(OUTPUT_ROOT / "attack_search_evaluation.json"),
    ])
    print("Checkpointed confirmatory workflow completed")